In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # nada sale a internet salvo Databricks si se configura
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))   # los notebooks importan de src/, nunca al revés

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

# 03 · Preprocesamiento — de las decisiones a código con pruebas

Cada decisión del EDA es una función de `src/proyecto/data/cleaning.py` con su prueba en `tests/test_cleaning.py`.
Este notebook **solo orquesta**. `data/raw/` no se modifica.

In [2]:
import json
import joblib
from proyecto.config import GROUP_COL, TARGET, PROCESSED_DIR
from proyecto.data.cleaning import cleaning_log, clean, personal_columns
from proyecto.data.split import split_by_group

raw = pd.read_csv(ROOT / "data" / "raw" / "Base_sintetica_creditos.csv", low_memory=False)
log = cleaning_log(raw)
clean_df = clean(raw)
print(json.dumps(log, indent=2, ensure_ascii=False))

{
  "filas_crudas": 60000,
  "columnas_crudas": 189,
  "columnas_personales_eliminadas": 49,
  "columnas_vacias_o_constantes": 63,
  "filas_bloque_reestructurado": 3619,
  "filas_sin_maduracion": 10331,
  "fecha_corte": "2026-10-30",
  "filas_finales": 46050,
  "clientes_finales": 34657,
  "tasa_mora_final": 0.15915309446254072,
  "columnas_finales": 17
}


## Comprobaciones (fallan si una decisión no se cumplió)

In [3]:
checks = {
    "sin datos personales": not personal_columns(clean_df.columns),
    "objetivo binario sin vacíos": clean_df[TARGET].isin([0, 1]).all(),
    "sin plazo de 1 mes (bloque reestructurado fuera)": (clean_df["term_months"] > 1).all(),
    "todas las filas tienen cliente": clean_df[GROUP_COL].notna().all(),
    "solo columnas conocidas al aprobar (17)": clean_df.shape[1] == 17,
    "edad en rango posible": clean_df["age"].dropna().between(18, 90).all(),
    "tasa EA en %": clean_df["interest_rate_ea"].between(5, 40).all(),
}
for k, v in checks.items():
    print("✔" if v else "✘", k)
assert all(checks.values())
clean_df.describe(include="all").T[["count", "unique", "mean", "min", "max"]]

✔ sin datos personales
✔ objetivo binario sin vacíos
✔ sin plazo de 1 mes (bloque reestructurado fuera)
✔ todas las filas tienen cliente
✔ solo columnas conocidas al aprobar (17)
✔ edad en rango posible
✔ tasa EA en %


,count,unique,mean,min,max
product,46050,7,NaN,NaN,NaN
term_months,46050.0,NaN,56.122671,3.0,240.0
loan_amount,46050.0,NaN,35736203.751661,150031.0,349949605.0
monthly_installment,46050.0,NaN,817108.281412,13409.0,5688599.0
interest_rate_ea,46050.0,NaN,20.196054,11.571884,28.575896
due_day,46050.0,NaN,11.576547,5.0,25.0
business_type,46050,3,NaN,NaN,NaN
economic_destination,46050,3,NaN,NaN,NaN
city,46050,22,NaN,NaN,NaN
age,46050.0,NaN,45.45101,19.0,72.0


## Partición por cliente

Un cliente tiene hasta 4 créditos: **todos sus créditos van al mismo lado**. 20 % de los clientes a prueba, semilla 42.
No hay orden temporal que respetar en la decisión (cada crédito se evalúa el día que se aprueba), pero las cohortes
recientes se analizan por separado en la validación.

In [4]:
train, test = split_by_group(clean_df, test_size=0.2)
print(f"entrenamiento: {len(train):,} créditos de {train[GROUP_COL].nunique():,} clientes · mora {train[TARGET].mean():.1%}")
print(f"prueba (CERRADA): {len(test):,} créditos de {test[GROUP_COL].nunique():,} clientes")
assert not set(train[GROUP_COL]) & set(test[GROUP_COL])
assert len(train) + len(test) == len(clean_df)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(train, PROCESSED_DIR / "train.joblib")
joblib.dump(test, PROCESSED_DIR / "test.joblib")   # no se vuelve a abrir hasta el notebook 07
log.update({"filas_entrenamiento": len(train), "filas_prueba": len(test),
            "clientes_entrenamiento": int(train[GROUP_COL].nunique()), "clientes_prueba": int(test[GROUP_COL].nunique()),
            "tasa_mora_entrenamiento": float(train[TARGET].mean())})
(ROOT / "reports" / "03_limpieza.json").write_text(json.dumps(log, indent=2, ensure_ascii=False), encoding="utf-8")
print("guardado en data/processed/ (no se versiona: se regenera con este notebook)")

entrenamiento: 36,857 créditos de 27,725 clientes · mora 16.1%
prueba (CERRADA): 9,193 créditos de 6,932 clientes
guardado en data/processed/ (no se versiona: se regenera con este notebook)


## Pruebas automáticas

```bash
pytest -q
```
Una prueba por regla de limpieza (`tests/test_cleaning.py`) y la de partición (`tests/test_split.py`, incluida una que
**falla** si alguien parte por fila).